In [1]:
pip install langchain openai faiss-cpu langchain-community sentence-transformers tqdm



╭──────────────────────────────────────────────────────────╮
│                                                          │
│  "As long as I’m alive, there are infinite chances.     │
│   You can’t give up. That’s what it means to be a pirate!"│
│                                                          │
│                     — Monkey D. Luffy                    │
╰──────────────────────────────────────────────────────────╯

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

# Load from the parquet files directly (dataset scripts are deprecated)
dataset = load_dataset(
    "ccdv/pubmed-summarization",
    split="train[:200]"
)


/home/amitdubey/Downloads/GEN-AI/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(dataset['article'][0])

a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . 
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . 
 the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . 
 anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . 
 snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states 
 there are also some reports regarding school feeding programs in developing countries . in 

In [4]:
print(dataset['abstract'][0])

background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and 2009 on 2897 primary and secondary school boys and girls ( 7 - 13 years old ) based on advocacy approach in shiraz , iran . 
 the project provided nutritious snacks in public schools over a 2-year period along with advocacy oriented actions in order to implement and promote nutritional intervention . for evaluation of effectiveness of the intervention growth monitoring indices of pre- and post - intervention were statistically compared.results:the frequency of subjects with body mass index lower than 5% decreased significantly after intervention among girls ( p = 0.02 ) . 
 however , there were no significant changes among boys or total population . 
 the mean of all anthropometric indices 

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline
import torch

# Using Phi-2, a 2.7B parameter model that's efficient and powerful
model_name = "microsoft/phi-2"

print(f"Loading LLM: {model_name}...")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Avoid padding warnings

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use FP16 for GPU efficiency
    device_map="auto",           # Automatically assign to GPU if available
    trust_remote_code=True
)

# Setup text-generation pipeline with max_new_tokens
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,  # Generate up to 256 new tokens
    do_sample=True,
    temperature=0.7,
    top_p=0.95
)
llm = HuggingFacePipeline(pipeline=llm_pipeline)

print(f"Model loaded successfully on device: {model.device}")


/home/amitdubey/Downloads/GEN-AI/.venv/lib64/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Loading LLM: microsoft/phi-2...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 453/453 [00:00<00:00, 533.71it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
Some parameters are on the meta device because they were offloaded to the cpu.
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded successfully on device: cuda:0


/tmp/ipykernel_92002/3559891257.py:31: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=llm_pipeline)


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_chunks = []

for item in dataset:
    # Combine abstract + article (or handle separately if you want)
    combined_text = item["abstract"] + "\n\n" + item["article"]
    chunks = splitter.create_documents([combined_text])
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
print(all_chunks[0].page_content[:500])  # print first 500 chars of first chunk


Total chunks: 18653
background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use a model like all-MiniLM-L6-v2
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name, 
                                   model_kwargs={"device": "cuda"})  # Use GPU since it's available

print(f"Embedding model loaded: {embedding_model_name}")


/tmp/ipykernel_92002/1158433412.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name,
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 841.48it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [8]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(all_chunks, embeddings)

In [9]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [10]:
from langchain_core.prompts import PromptTemplate

medical_template = """
You are a professional medical assistant AI. 

Guidelines:
- Use ONLY the provided text as context.
- If the information is not in the text, respond: "Not available in the provided context. Please consult a medical professional."
- Give concise, accurate, and neutral answers.
- Avoid guessing or providing unverified information.

Medical Context:
{context}

Patient Question:
{question}

Answer:
"""

medical_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=medical_template
)

print("Prompt template created successfully!")


Prompt template created successfully!


In [ ]:
# Integrated RAG + Summarization Chain
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Helper functions
def format_docs(docs):
    """Format retrieved documents into a single string"""
    return "\n\n".join(doc.page_content for doc in docs)

def extract_summary(text, num_sentences=3):
    """Extract first n sentences as summary"""
    sentences = text.split('.')
    summary = '. '.join(sentences[:num_sentences]) + '.' if sentences else text[:200]
    return summary

def format_for_summarization(retrieved_docs):
    """Combine and prepare docs for summarization"""
    combined = format_docs(retrieved_docs[:2])  # Top 2 docs
    return extract_summary(combined)

# Create unified pipeline: Query -> Retrieve -> Summarize -> Answer
unified_chain = (
    RunnableLambda(lambda x: x if isinstance(x, str) else x.get("question", x))
    | {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | medical_prompt
    | llm
    | StrOutputParser()
)

# Alternative: Retrieve -> Summarize Only (for quick summaries)
summary_only_chain = (
    RunnableLambda(lambda x: x if isinstance(x, str) else x.get("question", x))
    | retriever
    | RunnableLambda(format_for_summarization)
)



✓ Unified RAG Chain ready!
✓ Summary-Only Chain ready!


In [13]:
# Test both chains
test_query = "What are the symptoms and complications of diabetes?"

print("="*60)
print("UNIFIED RAG SYSTEM TEST")
print("="*60)
print(f"\nQuery: {test_query}\n")

# Test 1: Summary-only chain (fast)
print("--- SUMMARY OF RETRIEVED DOCUMENTS ---")
summary = summary_only_chain.invoke(test_query)
print(summary)

# Test 2: Full RAG chain (detailed answer)
print("\n--- FULL RAG ANSWER ---")
print("(Retrieving and generating detailed answer...)")
answer = unified_chain.invoke(test_query)
print(answer)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


UNIFIED RAG SYSTEM TEST

Query: What are the symptoms and complications of diabetes?

--- SUMMARY OF RETRIEVED DOCUMENTS ---
disease , and other chronic complications compared with individuals without diabetes. [18 ] chronic complications are the main cause of death among diabetic patients and account for the higher costs in hospitalization and drugs , and the costs of drugs for these complications are 2. 5 times higher

complications , community screening campaigns for diabetes , better diagnostic facilities specially in health centers and healthcare units , and better diabetes management systems and protocols .

--- FULL RAG ANSWER ---
(Retrieving and generating detailed answer...)

You are a professional medical assistant AI. 

Guidelines:
- Use ONLY the provided text as context.
- If the information is not in the text, respond: "Not available in the provided context. Please consult a medical professional."
- Give concise, accurate, and neutral answers.
- Avoid guessing or providing